# Empirical Analysis: Law & Crime Topic Prevalence Under Cross-National and Document-Type Shift

This notebook reproduces the ACS analysis structure for an LLM-based measurement task.
We use Llama 3.3 70B Instruct (4-bit quantized) to classify parliamentary and media texts
from the Comparative Agendas Project (CAP) as related to Law & Crime (CAP major topic
code 12), calibrate with Isotonic Regression and MCGrad on a balanced multi-country
sample, then measure prevalence estimation bias across a shift gradient from no shift
to maximum shift (unseen document type).

**Sub-populations (6 total):**

| Country | Doc type | Language | N (approx) | Role |
|---------|----------|----------|------------|------|
| Denmark | Parl. questions | Danish | 15K | Calibration + test |
| Spain | Oral questions | Spanish | 15K | Calibration + test |
| US | Congressional bills | English | 15K | Calibration + test |
| Belgium | Newspaper | Dutch | 15K | Calibration + test |
| Spain | Media (El Pais + El Mundo) | Spanish | 30K | OOD target |
| Belgium | TV news | Dutch | 15K | OOD target |

**Calibration design:** Balanced sample from 4 sub-populations (Denmark questions,
Spain questions, US bills, Belgium newspaper) gives MCGrad variation in country,
language, doc_type, decade, and party. All 4 countries and all 4 languages are
represented in calibration. OOD targets (Spain media, Belgium TV) share country
and language with calibration but introduce an unseen document type — mirroring
the realistic scenario of validating on one document type and applying to another.

In [ ]:
import ctypes
ctypes.cdll.LoadLibrary('/usr/lib64/libgomp.so.1')

import sys
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from mcgrad import methods as mcgrad_methods

sys.path.insert(0, "..")
from plot_config import METHOD_COLORS

os.makedirs('../paper/images', exist_ok=True)

plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 8,
    'figure.dpi': 150,
})

## 1. Data Loading

Load the full 105K sample (15K per sub-population across 7 sub-populations
in 4 countries) and join with Llama 3.3 70B inference scores.

In [ ]:
DATA_DIR = os.path.join('data')
LABEL_COLUMN = 'law_crime'
SCORE_COLUMN = 'llm_score'

# Load full sample and scores
data_df = pd.read_csv(os.path.join(DATA_DIR, 'full_sample.csv'))
scores_df = pd.read_csv(
    os.path.join(DATA_DIR, 'inference_output', 'llama-70b', 'full_scores.csv')
)

# Join scores positionally (same row order, verified during data prep)
data_df[SCORE_COLUMN] = scores_df['score'].astype(float)

# Linear squashing to keep scores away from 0/1 extremes.
# MCGrad's internal logit transform maps extreme values to +/-inf, causing
# 51% of scores to be clipped. Squashing to [eps, 1-eps] fixes this without
# altering rank ordering or introducing calibration assumptions.
EPSILON = 0.05
SQUASHED_COL = 'llm_score_squashed'
data_df[SQUASHED_COL] = EPSILON + (1 - 2 * EPSILON) * data_df[SCORE_COLUMN]

# Create subpop key matching the 7 sub-population structure
SUBPOP_MAP = {
    ('Denmark', 'parliamentary_question'): 'denmark_questions',
    ('Spain', 'parliamentary_question'): 'spain_questions',
    ('Spain', 'media'): 'spain_media',
    ('United States', 'bill'): 'us_bills',
    ('Belgium', 'tv_news'): 'belgium_tv',
    ('Belgium', 'newspaper'): 'belgium_newspaper',
}
data_df['subpop'] = data_df.apply(
    lambda r: SUBPOP_MAP.get((r['country'], r['doc_type']), 'unknown'), axis=1
)

# Split into sub-population DataFrames
subpops = {}
for key in SUBPOP_MAP.values():
    sub = data_df[data_df['subpop'] == key].copy()
    subpops[key] = sub

# Spain media: El Pais and El Mundo are combined in one subpop key
# but we keep them together since they share country+doc_type

print("Sub-population summary:")
print(f"{'Key':<25} {'N':>7} {'Prevalence':>10} {'Mean Score':>10} {'Mean Squashed':>14}")
print("-" * 70)
for key, df in subpops.items():
    print(f"{key:<25} {len(df):>7,} {df[LABEL_COLUMN].mean():>9.1%} "
          f"{df[SCORE_COLUMN].mean():>10.3f} {df[SQUASHED_COL].mean():>13.3f}")
print(f"\nTotal: {len(data_df):,} documents")
print(f"Squashing: score_new = {EPSILON} + {1-2*EPSILON} * score_raw  =>  [{EPSILON}, {1-EPSILON}]")

In [ ]:
from sklearn.metrics import roc_auc_score

print("=== LLM discriminative performance (AUC) by sub-population ===")
for key, df in subpops.items():
    auc = roc_auc_score(df[LABEL_COLUMN], df[SCORE_COLUMN])
    pos_mean = df.loc[df[LABEL_COLUMN] == 1, SCORE_COLUMN].mean()
    neg_mean = df.loc[df[LABEL_COLUMN] == 0, SCORE_COLUMN].mean()
    print(f"  {key:<25} AUC={auc:.3f}  pos_mean={pos_mean:.3f}  neg_mean={neg_mean:.3f}")

print("\n=== Year range by sub-population ===")
for key, df in subpops.items():
    print(f"  {key}: {df['year'].min()}--{df['year'].max()}")

print("\n=== Party coverage ===")
for key, df in subpops.items():
    if 'party' in df.columns and df['party'].notna().any():
        n_parties = df['party'].nunique()
        coverage = df['party'].notna().mean()
        print(f"  {key}: {n_parties} parties, {coverage:.0%} coverage")
    else:
        print(f"  {key}: no party data")

### Score Distribution Diagnostic

The LLM produces log-probability-based scores via P(Yes) / (P(Yes) + P(No)).
These are not calibrated posteriors — they tend to be **bimodal** (near 0 or near 1),
with a substantial fraction of negatives receiving very high scores. This has
important implications for all downstream methods.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
subpop_order = ['denmark_questions', 'spain_questions', 'us_bills',
                'belgium_newspaper', 'spain_media', 'belgium_tv']
titles = ['Denmark Questions\n(calibration)', 'Spain Questions\n(calibration)',
          'US Bills\n(calibration)', 'Belgium Newspaper\n(calibration)',
          'Spain Media\n(OOD)', 'Belgium TV\n(OOD)']

for ax, key, title in zip(axes.flat, subpop_order, titles):
    df = subpops[key]
    neg_scores = df.loc[df[LABEL_COLUMN] == 0, SCORE_COLUMN]
    pos_scores = df.loc[df[LABEL_COLUMN] == 1, SCORE_COLUMN]

    ax.hist(neg_scores, bins=50, alpha=0.6, color='steelblue',
            label=f'Y=0 (n={len(neg_scores):,})', density=True)
    ax.hist(pos_scores, bins=50, alpha=0.6, color='coral',
            label=f'Y=1 (n={len(pos_scores):,})', density=True)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('LLM Score')
    ax.legend(fontsize=6)

    # Show fraction of extreme scores
    high_neg = (neg_scores > 0.99).mean()
    high_pos = (pos_scores > 0.99).mean()
    low_neg = (neg_scores < 0.01).mean()
    ax.text(0.5, 0.95,
            f'Y=0 > 0.99: {high_neg:.1%}\nY=1 > 0.99: {high_pos:.1%}\nY=0 < 0.01: {low_neg:.1%}',
            transform=ax.transAxes, fontsize=6, va='top', ha='center',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.suptitle('LLM Score Distribution by Label and Sub-population', fontsize=13)
fig.tight_layout()
fig.savefig(
    '../paper/images/figure_cap_score_distribution.png',
    dpi=300, bbox_inches='tight',
)
plt.show()

# Aggregate extreme value statistics
print("\n=== Extreme value analysis ===")
print(f"{'Subpop':<25} {'%neg>0.99':>10} {'%neg<0.01':>10} {'%pos>0.99':>10} {'%pos<0.01':>10}")
print("-" * 70)
for key in subpop_order:
    df = subpops[key]
    neg = df.loc[df[LABEL_COLUMN] == 0, SCORE_COLUMN]
    pos = df.loc[df[LABEL_COLUMN] == 1, SCORE_COLUMN]
    print(f"{key:<25} {(neg > 0.99).mean():>9.1%} {(neg < 0.01).mean():>9.1%} "
          f"{(pos > 0.99).mean():>9.1%} {(pos < 0.01).mean():>9.1%}")

## 2. Calibration Setup

Calibrate on a **balanced sample from 4 sub-populations** to give MCGrad
variation in country, language, doc_type, decade, and party:
- Denmark questions (~15K)
- Spain questions (~15K)
- US bills (~15K)
- Belgium newspaper (~15K)

Total calibration: ~18K (4.5K per sub-population). The remaining data from
these 4 sub-populations forms the in-distribution test set.

All 4 countries and all 4 languages are represented in calibration.

**OOD targets** (share country and language with calibration, unseen doc_type):
- Spain media (El Pais + El Mundo) -- same country/language, new doc type
- Belgium TV news -- same country/language, new doc type

In [ ]:
# Build calibration set: balanced sample from 4 sub-populations
CAL_SUBPOPS = ['denmark_questions', 'spain_questions', 'us_bills', 'belgium_newspaper']
CAL_N_PER_SUBPOP = 15_000

calibration_parts = []
test_parts = []

for key in CAL_SUBPOPS:
    df = subpops[key].copy()
    # Target ~4500 per subpop to keep total calibration ~18K
    cal_frac = min(4_500 / len(df), 0.40)
    cal_part, test_part = train_test_split(
        df,
        test_size=1.0 - cal_frac,
        random_state=42,
        stratify=df[LABEL_COLUMN],
    )
    calibration_parts.append(cal_part)
    test_parts.append(test_part)
    print(f"  {key}: {len(cal_part):,} calibration, {len(test_part):,} test")

calibration_df = pd.concat(calibration_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)

print(f"\nCalibration set: {len(calibration_df):,} samples")
print(f"  Law & Crime prevalence: {calibration_df[LABEL_COLUMN].mean():.1%}")
print(f"  By country: {calibration_df['country'].value_counts().to_dict()}")
print(f"  By doc_type: {calibration_df['doc_type'].value_counts().to_dict()}")

print(f"\nIn-distribution test set: {len(test_df):,} samples")
print(f"  Law & Crime prevalence: {test_df[LABEL_COLUMN].mean():.1%}")

In [ ]:
# --- Feature preparation ---
# Create decade feature and fill missing party for all DataFrames

all_dfs_to_prep = [calibration_df, test_df] + [
    subpops[k] for k in subpops if k not in CAL_SUBPOPS
]
for df in all_dfs_to_prep:
    df['decade'] = (df['year'] // 10) * 10
    # Fill missing party with 'unknown' for MCGrad compatibility
    if 'party' in df.columns:
        df['party'] = df['party'].fillna('unknown')
    else:
        df['party'] = 'unknown'

In [ ]:
# --- Fit calibration methods ---

# 1. Isotonic Regression (global calibration) — fitted on raw scores
isotonic_reg = mcgrad_methods.IsotonicRegression().fit(
    calibration_df,
    SCORE_COLUMN,
    LABEL_COLUMN,
)
print("Isotonic regression fitted (on raw scores)")

# 2. MCGrad (multicalibration) — fitted on squashed scores
# Squashing maps [0,1] -> [0.05, 0.95] so MCGrad's logit transform
# doesn't clip extreme values.
CATEGORICAL_SEGMENT_FEATURES = ['doc_type', 'country', 'party']
NUMERICAL_SEGMENT_FEATURES = ['decade']

mcgrad = mcgrad_methods.MCGrad()
mcgrad = mcgrad.fit(
    calibration_df,
    SQUASHED_COL,
    LABEL_COLUMN,
    categorical_feature_column_names=CATEGORICAL_SEGMENT_FEATURES,
    numerical_feature_column_names=NUMERICAL_SEGMENT_FEATURES,
)
print("MCGrad fitted (on squashed scores)")

In [ ]:
# Generate calibrated predictions for test set and all OOD sub-populations
IR_COL = 'isotonic_prediction'
MCGRAD_COL = 'mcgrad_prediction'

# OOD sub-populations (not part of calibration split)
OOD_KEYS = [k for k in subpops if k not in CAL_SUBPOPS]

eval_dfs = [test_df] + [subpops[k] for k in OOD_KEYS]

for df in eval_dfs:
    # Isotonic regression predictions (from raw scores)
    df[IR_COL] = isotonic_reg.predict(df, SCORE_COLUMN)

    # MCGrad predictions (from squashed scores)
    df[MCGRAD_COL] = mcgrad.predict(
        df=df,
        prediction_column_name=SQUASHED_COL,
        categorical_feature_column_names=CATEGORICAL_SEGMENT_FEATURES,
        numerical_feature_column_names=NUMERICAL_SEGMENT_FEATURES,
    )

print("Calibrated predictions generated for all evaluation datasets")

In [ ]:
# --- Calibration parameters for quantification methods ---
from sklearn.metrics import roc_curve


def calibrate_threshold_youden(labels, predictions):
    """Find threshold that maximizes Youden's J = TPR - FPR."""
    fpr_arr, tpr_arr, thresholds = roc_curve(labels, predictions)
    j_scores = tpr_arr - fpr_arr
    best_idx = np.argmax(j_scores)
    return float(thresholds[best_idx])


def estimate_classifier_error_rates(labels, predictions, threshold):
    """Estimate TPR and FPR from calibration data at given threshold."""
    binary_preds = (predictions >= threshold).astype(int)
    labels_arr = labels.astype(int)
    tp = ((binary_preds == 1) & (labels_arr == 1)).sum()
    fp = ((binary_preds == 1) & (labels_arr == 0)).sum()
    tn = ((binary_preds == 0) & (labels_arr == 0)).sum()
    fn = ((binary_preds == 0) & (labels_arr == 1)).sum()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    return tpr, fpr


THRESHOLD = calibrate_threshold_youden(
    labels=calibration_df[LABEL_COLUMN],
    predictions=calibration_df[SCORE_COLUMN],
)
print(f"Calibrated threshold (Youden's J): {THRESHOLD:.4f}")

calibration_tpr, calibration_fpr = estimate_classifier_error_rates(
    labels=calibration_df[LABEL_COLUMN],
    predictions=calibration_df[SCORE_COLUMN],
    threshold=THRESHOLD,
)
print(f"\nCalibration set error rates at threshold={THRESHOLD:.4f}:")
print(f"  TPR (sensitivity): {calibration_tpr:.4f}")
print(f"  FPR (1-specificity): {calibration_fpr:.4f}")

# PACC parameters
PACC_POS_MEAN = calibration_df[calibration_df[LABEL_COLUMN] == 1][SCORE_COLUMN].mean()
PACC_NEG_MEAN = calibration_df[calibration_df[LABEL_COLUMN] == 0][SCORE_COLUMN].mean()
SOURCE_PREVALENCE = calibration_df[LABEL_COLUMN].mean()

print(f"\nPACC parameters (soft-score Rogan-Gladen):")
print(f"  E[h(X)|Y=1]: {PACC_POS_MEAN:.4f}")
print(f"  E[h(X)|Y=0]: {PACC_NEG_MEAN:.4f}")
print(f"\nSource prevalence for SLD: {SOURCE_PREVALENCE:.4f}")

# Verify
apparent_prev = (calibration_df[SCORE_COLUMN] >= THRESHOLD).mean()
true_prev = calibration_df[LABEL_COLUMN].mean()
print(f"\nVerification:")
print(f"  True prevalence in calibration set: {true_prev:.4f}")
print(f"  Apparent prevalence at threshold: {apparent_prev:.4f}")

## 3. Shift Gradient Construction

We construct a gradient of increasing distributional shift:

| Scenario | Source | Shift type |
|----------|--------|------------|
| Baseline | Balanced test split (same composition as calibration) | None |
| Country shift | Test split resampled to overweight Belgium | Within-calibration |
| Doc-type shift | Test split resampled to overweight US bills | Within-calibration |
| Party shift | Test split resampled to overweight one party | Within-calibration |
| OOD: Spain media | Spain media (El Pais + El Mundo) | Same country/lang, new doc type |
| OOD: Belgium TV | Belgium TV news | Same country/lang, new doc type |

In [ ]:
def resample_with_party_shift(
    df,
    shift='original',
    n_samples=20_000,
    random_state=42,
):
    actual_n = min(n_samples, len(df))
    if shift == 'original':
        weights = np.ones(len(df))
    else:
        party_topic_rate = df.groupby('party')[LABEL_COLUMN].mean()
        party_rank = party_topic_rate.rank(pct=True)
        rank_values = df['party'].map(party_rank).values

        if shift == 'left_heavy':
            weights = np.exp(-2.0 * rank_values)
        elif shift == 'right_heavy':
            weights = np.exp(2.0 * rank_values)
        elif shift == 'polarized':
            weights = np.exp(2.0 * np.abs(rank_values - 0.5))
        else:
            raise ValueError(f"Unknown shift: {shift}")

    weights = weights / weights.sum()
    return df.sample(
        n=actual_n,
        weights=weights,
        replace=True,
        random_state=random_state,
    )


def resample_with_doctype_shift(
    df,
    target_doctype='bill',
    overweight_factor=5.0,
    n_samples=20_000,
    random_state=42,
):
    actual_n = min(n_samples, len(df))
    weights = np.where(
        df['doc_type'] == target_doctype,
        overweight_factor,
        1.0,
    )
    weights = weights / weights.sum()
    return df.sample(
        n=actual_n,
        weights=weights,
        replace=True,
        random_state=random_state,
    )


def resample_with_country_shift(
    df,
    target_country='Belgium',
    overweight_factor=5.0,
    n_samples=20_000,
    random_state=42,
):
    actual_n = min(n_samples, len(df))
    weights = np.where(
        df['country'] == target_country,
        overweight_factor,
        1.0,
    )
    weights = weights / weights.sum()
    return df.sample(
        n=actual_n,
        weights=weights,
        replace=True,
        random_state=random_state,
    )


N_SAMPLES = 20_000

scenarios = {
    'Baseline\n(balanced test)': {
        'df': test_df.sample(
            n=min(N_SAMPLES, len(test_df)),
            replace=True, random_state=42),
        'shift_type': 'none',
    },
    'Country shift\n(overweight Belgium)': {
        'df': resample_with_country_shift(
            test_df, target_country='Belgium',
            n_samples=N_SAMPLES, random_state=42),
        'shift_type': 'within-calibration',
    },
    'Doc-type shift\n(overweight bills)': {
        'df': resample_with_doctype_shift(
            test_df, target_doctype='bill',
            n_samples=N_SAMPLES, random_state=42),
        'shift_type': 'within-calibration',
    },
    'Party shift\n(right-heavy)': {
        'df': resample_with_party_shift(
            test_df, shift='right_heavy',
            n_samples=N_SAMPLES, random_state=42),
        'shift_type': 'within-calibration',
    },
    'Spain media\n(OOD doc type)': {
        'df': subpops['spain_media'].sample(
            n=min(N_SAMPLES, len(subpops['spain_media'])),
            replace=True, random_state=42),
        'shift_type': 'OOD (new doc type)',
    },
    'Belgium TV\n(OOD doc type)': {
        'df': subpops['belgium_tv'].sample(
            n=min(N_SAMPLES, len(subpops['belgium_tv'])),
            replace=True, random_state=42),
        'shift_type': 'OOD (new doc type)',
    },
}

print("Shift gradient scenarios:")
print(f"{'Scenario':<35} {'N':>7}  {'True prev':>10}  {'Shift type'}")
print("-" * 80)
for label, info in scenarios.items():
    df = info['df']
    clean_label = label.replace('\n', ' ')
    print(f"{clean_label:<35} {len(df):>7,}  "
          f"{df[LABEL_COLUMN].mean():>9.1%}  {info['shift_type']}")

## 4. Prevalence Estimation

For each scenario along the shift gradient, compute prevalence estimates
with all 7 methods:
1. **Raw LLM scores** (mean of uncalibrated scores)
2. **Classify & Count** (fraction above threshold)
3. **Rogan-Gladen** (CC adjusted by TPR/FPR)
4. **PACC** (soft-score Rogan-Gladen)
5. **SLD (EMQ)** (EM algorithm)
6. **Isotonic Regression** (mean of isotonic-calibrated scores)
7. **MCGrad** (mean of multicalibration-adjusted scores)

In [ ]:
def sld_estimate(scores, source_prevalence, max_iter=100, tol=1e-6):
    """Saerens-Latinne-Decaestecker (EMQ) prevalence estimator.

    EM algorithm that iteratively re-estimates prevalence by adjusting
    posteriors for a new prior. Assumes label shift (P(X|Y) stable).
    """
    p_hat = source_prevalence
    for _ in range(max_iter):
        ratio_pos = p_hat / source_prevalence
        ratio_neg = (1 - p_hat) / (1 - source_prevalence)
        adjusted = (ratio_pos * scores) / (ratio_pos * scores + ratio_neg * (1 - scores))
        p_new = adjusted.mean()
        if abs(p_new - p_hat) < tol:
            break
        p_hat = p_new
    return p_hat


def pacc_estimate(scores, pos_mean, neg_mean):
    """Probabilistic Adjusted Classify & Count.

    Soft-score generalization of Rogan-Gladen: uses E[h(X)|Y=1] and E[h(X)|Y=0]
    instead of binary TPR/FPR.
    """
    pcc = scores.mean()
    denom = pos_mean - neg_mean
    if abs(denom) < 1e-10:
        return pcc
    return np.clip((pcc - neg_mean) / denom, 0.0, 1.0)


def compute_rogan_gladen_estimate(apparent_prevalence, tpr, fpr):
    """Rogan-Gladen adjusted prevalence estimate."""
    denominator = tpr - fpr
    if abs(denominator) < 1e-10:
        return apparent_prevalence
    adjusted = (apparent_prevalence - fpr) / denominator
    return max(0.0, min(1.0, adjusted))


def compute_all_prevalence_estimates(
    target_df,
    score_col,
    ir_col,
    mcgrad_col,
    label_col,
    cal_tpr,
    cal_fpr,
    pacc_pos_mean,
    pacc_neg_mean,
    source_prevalence,
    threshold=0.5,
):
    """Compute prevalence estimates using all 7 methods."""
    true_prevalence = target_df[label_col].mean()

    # 1. Raw LLM scores
    raw_estimate = target_df[score_col].mean()

    # 2. Classify & Count
    binary_preds = (target_df[score_col] >= threshold).astype(int)
    classify_count = binary_preds.mean()

    # 3. Rogan-Gladen
    rogan_gladen = compute_rogan_gladen_estimate(classify_count, cal_tpr, cal_fpr)

    # 4. PACC
    pacc = pacc_estimate(target_df[score_col].values, pacc_pos_mean, pacc_neg_mean)

    # 5. SLD (EMQ)
    sld = sld_estimate(target_df[score_col].values, source_prevalence)

    # 6. Isotonic Regression
    isotonic_estimate = target_df[ir_col].mean()

    # 7. MCGrad
    mcgrad_estimate = target_df[mcgrad_col].mean()

    return {
        'True Prevalence': true_prevalence,
        'Raw Scores': raw_estimate,
        'Classify & Count': classify_count,
        'Rogan-Gladen': rogan_gladen,
        'PACC': pacc,
        'SLD (EMQ)': sld,
        'Isotonic Regression': isotonic_estimate,
        'MCGrad': mcgrad_estimate,
    }


def compute_bias_table(estimates):
    """Create a DataFrame showing estimates and bias for each method."""
    true_prev = estimates['True Prevalence']
    rows = []
    for method, estimate in estimates.items():
        if method == 'True Prevalence':
            continue
        bias = estimate - true_prev
        rows.append({
            'Method': method,
            'Estimate': estimate,
            'Bias': bias,
            'Relative Bias (%)': 100 * bias / true_prev if true_prev > 0 else 0,
        })
    return pd.DataFrame(rows).set_index('Method')


# Methods to compare (same as ACS)
methods_to_plot = [
    'Raw Scores', 'Classify & Count', 'Rogan-Gladen', 'PACC',
    'SLD (EMQ)', 'Isotonic Regression', 'MCGrad',
]

colors_map = {m: METHOD_COLORS[m] for m in methods_to_plot}

In [ ]:
# Compute prevalence estimates for each scenario along the shift gradient
all_results = {}

for scenario_label, info in scenarios.items():
    syn_df = info['df']
    estimates = compute_all_prevalence_estimates(
        target_df=syn_df,
        score_col=SCORE_COLUMN,
        ir_col=IR_COL,
        mcgrad_col=MCGRAD_COL,
        label_col=LABEL_COLUMN,
        cal_tpr=calibration_tpr,
        cal_fpr=calibration_fpr,
        pacc_pos_mean=PACC_POS_MEAN,
        pacc_neg_mean=PACC_NEG_MEAN,
        source_prevalence=SOURCE_PREVALENCE,
        threshold=THRESHOLD,
    )
    all_results[scenario_label] = estimates
    clean_label = scenario_label.replace('\n', ' ')
    print(f"\n{clean_label}:")
    print(f"  True prevalence: {estimates['True Prevalence']:.4f}")
    display(compute_bias_table(estimates).round(4))

## 5. Bootstrap RMSE

200 bootstrap iterations per scenario. For each iteration, resample with
replacement from the scenario's source population and compute prevalence
estimates. Report bias and RMSE across resamples.

Bootstrap sample size: `min(len(source_population), 20_000)` to handle
sub-populations smaller than 20K (e.g., Belgium newspaper ~21K).

In [ ]:
def compute_bootstrap_rmse(
    source_df,
    score_col,
    ir_col,
    mcgrad_col,
    label_col,
    cal_tpr,
    cal_fpr,
    pacc_pos_mean,
    pacc_neg_mean,
    source_prevalence,
    threshold,
    n_bootstrap=200,
    n_samples=20_000,
):
    """Bootstrap resampling to compute bias, variance, and RMSE.

    Resamples uniformly from the source population (no additional shift).
    Sample size is min(n_samples, len(source_df)).
    """
    actual_n = min(n_samples, len(source_df))
    methods_list = [
        'Raw Scores', 'Classify & Count', 'Rogan-Gladen', 'PACC',
        'SLD (EMQ)', 'Isotonic Regression', 'MCGrad',
    ]
    estimates_by_method = {m: [] for m in methods_list}
    true_prevs = []

    for b in range(n_bootstrap):
        syn_df = source_df.sample(
            n=actual_n,
            replace=True,
            random_state=b,
        )
        est = compute_all_prevalence_estimates(
            syn_df, score_col, ir_col, mcgrad_col, label_col,
            cal_tpr, cal_fpr, pacc_pos_mean, pacc_neg_mean,
            source_prevalence, threshold,
        )
        true_prevs.append(est['True Prevalence'])
        for m in methods_list:
            estimates_by_method[m].append(est[m])

    results = {}
    for m in methods_list:
        ests = np.array(estimates_by_method[m])
        trues = np.array(true_prevs)
        errors = ests - trues
        results[m] = {
            'bias': np.mean(errors) * 100,       # in percentage points
            'variance': np.var(errors) * 100**2,  # in pp^2
            'rmse': np.sqrt(np.mean(errors**2)) * 100,  # in pp
        }
    results['True Prevalence'] = np.mean(true_prevs)
    return results


def compute_bootstrap_rmse_shifted(
    source_df,
    shift_fn,
    shift_kwargs,
    score_col,
    ir_col,
    mcgrad_col,
    label_col,
    cal_tpr,
    cal_fpr,
    pacc_pos_mean,
    pacc_neg_mean,
    source_prevalence,
    threshold,
    n_bootstrap=200,
):
    """Bootstrap with a shift resampling function (for within-calibration scenarios)."""
    methods_list = [
        'Raw Scores', 'Classify & Count', 'Rogan-Gladen', 'PACC',
        'SLD (EMQ)', 'Isotonic Regression', 'MCGrad',
    ]
    estimates_by_method = {m: [] for m in methods_list}
    true_prevs = []

    for b in range(n_bootstrap):
        syn_df = shift_fn(source_df, random_state=b, **shift_kwargs)
        est = compute_all_prevalence_estimates(
            syn_df, score_col, ir_col, mcgrad_col, label_col,
            cal_tpr, cal_fpr, pacc_pos_mean, pacc_neg_mean,
            source_prevalence, threshold,
        )
        true_prevs.append(est['True Prevalence'])
        for m in methods_list:
            estimates_by_method[m].append(est[m])

    results = {}
    for m in methods_list:
        ests = np.array(estimates_by_method[m])
        trues = np.array(true_prevs)
        errors = ests - trues
        results[m] = {
            'bias': np.mean(errors) * 100,
            'variance': np.var(errors) * 100**2,
            'rmse': np.sqrt(np.mean(errors**2)) * 100,
        }
    results['True Prevalence'] = np.mean(true_prevs)
    return results

In [ ]:
print("Computing bootstrap RMSE (200 resamples per scenario)...")

bootstrap_config = {
    'Baseline\n(balanced test)': {
        'mode': 'uniform', 'source': test_df,
    },
    'Country shift\n(overweight Belgium)': {
        'mode': 'shift', 'source': test_df,
        'shift_fn': resample_with_country_shift,
        'shift_kwargs': {'target_country': 'Belgium', 'n_samples': N_SAMPLES},
    },
    'Doc-type shift\n(overweight bills)': {
        'mode': 'shift', 'source': test_df,
        'shift_fn': resample_with_doctype_shift,
        'shift_kwargs': {'target_doctype': 'bill', 'n_samples': N_SAMPLES},
    },
    'Party shift\n(right-heavy)': {
        'mode': 'shift', 'source': test_df,
        'shift_fn': resample_with_party_shift,
        'shift_kwargs': {'shift': 'right_heavy', 'n_samples': N_SAMPLES},
    },
    'Spain media\n(OOD doc type)': {
        'mode': 'uniform', 'source': subpops['spain_media'],
    },
    'Belgium TV\n(OOD doc type)': {
        'mode': 'uniform', 'source': subpops['belgium_tv'],
    },
}

common_args = dict(
    score_col=SCORE_COLUMN, ir_col=IR_COL, mcgrad_col=MCGRAD_COL,
    label_col=LABEL_COLUMN, cal_tpr=calibration_tpr, cal_fpr=calibration_fpr,
    pacc_pos_mean=PACC_POS_MEAN, pacc_neg_mean=PACC_NEG_MEAN,
    source_prevalence=SOURCE_PREVALENCE, threshold=THRESHOLD,
)

bootstrap_results = {}
for label, cfg in bootstrap_config.items():
    clean_label = label.replace('\n', ' ')
    print(f"  {clean_label}...")
    if cfg['mode'] == 'shift':
        bootstrap_results[label] = compute_bootstrap_rmse_shifted(
            cfg['source'],
            shift_fn=cfg['shift_fn'],
            shift_kwargs=cfg['shift_kwargs'],
            **common_args,
        )
    else:
        bootstrap_results[label] = compute_bootstrap_rmse(
            cfg['source'], **common_args,
        )

print("Done.")

## 6. Results Table

Summary table showing bias and RMSE across the shift gradient,
mirroring the ACS Table 1 format. Rows ordered from no shift to maximum shift.

In [ ]:
# Summary table: bias and RMSE across shift gradient
shift_summary = []

scenario_order = list(scenarios.keys())

for scenario_label in scenario_order:
    estimates = all_results[scenario_label]
    bs = bootstrap_results[scenario_label]
    true_prev = estimates['True Prevalence']
    clean_label = scenario_label.replace('\n', ' ')
    shift_type = scenarios[scenario_label]['shift_type']
    row = {
        'Scenario': clean_label,
        'Shift Type': shift_type,
        'True Prevalence': f'{true_prev:.1%}',
    }
    for method in methods_to_plot:
        bias_pp = (estimates[method] - true_prev) * 100
        rmse_pp = bs[method]['rmse']
        row[f'{method} Bias'] = f'{bias_pp:+.2f}pp'
        row[f'{method} RMSE'] = f'{rmse_pp:.2f}pp'
    shift_summary.append(row)

summary_df = pd.DataFrame(shift_summary).set_index(['Scenario', 'Shift Type'])
print('=== Law & Crime Prevalence Estimation: Bias and RMSE Across Shift Gradient ===\n')
summary_df

## 7. Figure: Bias Across the Shift Gradient

Single-panel bar chart showing prevalence estimation bias for each method
across the shift gradient (from baseline to maximum shift).
Uses `METHOD_COLORS` from `plot_config.py`, same style as ACS Figure 2.

In [ ]:
scenario_labels = list(scenarios.keys())

fig, ax = plt.subplots(figsize=(14, 6.5))

x = np.arange(len(scenario_labels))
n_methods = len(methods_to_plot)
group_width = 0.82
bar_width = group_width / n_methods * 0.85

for i, method in enumerate(methods_to_plot):
    biases = [
        (all_results[s][method] - all_results[s]['True Prevalence']) * 100
        for s in scenario_labels
    ]
    offset = (i - n_methods / 2 + 0.5) * (group_width / n_methods)
    ax.bar(
        x + offset, biases, bar_width,
        label=method, color=colors_map[method],
        edgecolor='white', linewidth=0.5,
    )

ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('Bias (percentage points)')
ax.set_xticks(x)
ax.set_xticklabels(scenario_labels, rotation=45, ha='right')
ax.legend(fontsize=7, loc='best')

# Add shift gradient annotation
ax.annotate(
    '', xy=(len(scenario_labels) - 0.5, ax.get_ylim()[0]),
    xytext=(-0.5, ax.get_ylim()[0]),
    arrowprops=dict(arrowstyle='->', color='#888888', lw=1.5),
)
ax.text(
    len(scenario_labels) / 2 - 0.5, ax.get_ylim()[0] * 0.95,
    'increasing distributional shift $\\longrightarrow$',
    ha='center', va='top', fontsize=9, color='#888888', style='italic',
)

fig.suptitle(
    'Law & Crime Prevalence Estimation Bias Across Shift Gradient',
    fontsize=13,
)
fig.tight_layout()
fig.savefig(
    '../paper/images/figure_cap_shift_gradient.png',
    dpi=300, bbox_inches='tight',
)
plt.show()